In [1]:
from sklearn.linear_model import LogisticRegression
from src.data import load_data, split_by_year
from src.features import add_ratios
from src.config import RISK_TREND
from src.metrics import evaluate
from src.scorecard import WOEBinner

In [2]:
train, val, test = (add_ratios(d) for d in split_by_year(load_data()))

In [3]:
# 1. bins from train only, then check them
binner = WOEBinner(RISK_TREND).fit(train, train["default"])
binner.iv()

mve_tl    1.327459
tl_ta     1.047409
quick     0.836434
ni_ta     0.769016
re_ta     0.538928
log_ta    0.087330
dtype: float64

In [4]:
W_tr, W_va, W_te = (binner.transform(d) for d in (train, val, test))

In [5]:
lr = LogisticRegression(max_iter = 1000).fit(W_tr, train["default"])

dict(zip(W_tr.columns, lr.coef_[0]))

{'mve_tl': np.float64(-0.6870232946872344),
 'tl_ta': np.float64(-0.11920438006153927),
 'ni_ta': np.float64(-0.5952907017744155),
 'quick': np.float64(-0.34110061073572967),
 're_ta': np.float64(-0.19038684264082814),
 'log_ta': np.float64(0.0033624237433481537)}

In [6]:
import pandas as pd
pd.DataFrame({
    "train": evaluate(train["default"], lr.predict_proba(W_tr)[:,1]),
    "val":evaluate(val["default"], lr.predict_proba(W_va)[:,1]),
    "test":evaluate(test["default"], lr.predict_proba(W_te)[:,1]),
})

,train,val,test
n,55927.000000,10473.000000,12282.000000
n_pos,403.000000,87.000000,119.000000
base_rate,0.007206,0.008307,0.009689
pr_auc,0.045246,0.076422,0.103145
pr_auc_lift,6.279108,9.199576,10.645565
roc_auc,0.850303,0.909293,0.906214
brier,0.006988,0.007925,0.009160
brier_skill,0.023221,0.038052,0.045398
mean_pred,0.007204,0.006467,0.006901
ks,0.575751,0.684380,0.679123
